In [ ]:
input_data = None
targetpop_data = None
output_data = None
output_model = None
util = None
display_util = None
configfile = "config/config.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import matplotlib.pyplot as plt

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)
plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rule_setup,
    display_data_doc,
    display_long_data_doc,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    SpenderID,
    drop_duplicate_columns,
    common_translate,
    split_data,
    collapse_col,
    find_redundant_cols,
    fix_redundancies,
)

### Target Population Filtering

The donors in the dataset were filtered to match the target population (see [](general:tpf)). Afterwards we tried again to remove empty and duplicate columns.

In [ ]:
data = pd.read_parquet(input_data)
donors = collapse_col(
    data.loc[:, ["donor_et_dso", "donor_et_id_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
targetpop = pd.read_parquet(targetpop_data)
data = data[donors.isin(targetpop["donor_et_id_et"])]
display(
    Markdown(
        f"""The filter process reduced the number of donors in the data ({donors.nunique()}) and target population ({targetpop["donor_et_id_et"].nunique()})
            to {donors[donors.isin(targetpop["donor_et_id_et"])].nunique()} in the processed data.
        """
    )
)

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

### Integration of Seperated Institute Data

In this file, the {term}`DSO` and {term}`ET` data is sharing one row (see [](general:ic)) if the examination was taken on the same day. {term}`ET` only contributed a examination date and the type of examination.

In [ ]:
idcols = ["donor_et_dso", "donor_et_id_et"]
assert len(split_data(data, idcols)) == 3, "Not 3 different row types present!?"
assert (
    data.loc[:, ["examination_date_et", "examination_date"]]
    .diff(axis=1)
    .iloc[:, 1]
    .dropna()
    == 0
).all(), "Dates sometimes different"

## Domain Steps

For this file the general plan for domain preprocessing of longitudinal data was followed (see [](general:ds)).

### Row Filtering

There is no column differentiating between different types of tests (see [](general:rf)). The following analysis compares the data from the different sources. 

In [ ]:
data["Institute with a examination date"] = (
    (~data["examination_date"].isna()) + (~data["examination_date_et"].isna()) * 2
).replace({1: "DSO", 2: "ET", 3: "DSO+ET", 0: "No Date"})
data["donor"] = donors[donors.isin(targetpop["donor_et_id_et"])]
data["examination_date_both"] = collapse_col(
    data.loc[:, ["examination_date", "examination_date_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
display_long_data_doc(
    data,
    [
        "donor",
    ],
    "examination_date_both",
    "Institute with a examination date",
)
data.drop(
    columns=["examination_date_both", "donor", "Institute with a examination date"],
    inplace=True,
)

The rows and columns from only the {term}`ET` were dropped, as they contained only examination type data which is in another language as the {term}`DSO` exam type.

In [ ]:
data = data[~data["examination_date"].isna()].copy().drop(columns="type_et")

### Unit Conversions

We applied the common translations (see [](general:uc)).

In [ ]:
data = common_translate(data, config["data"]["common_translations"])

### Consolidating Columns

Consolidation removed the columns from the {term}`ET` (see [](general:crc)).

In [ ]:
red = find_redundant_cols(data)
# you can manually add if necessary
red["donor_et_id_et"] = ["donor_et_dso", "donor_et_id_et"]
fix_redundancies(data, red)

## Intermediate Dataset

For this longitudinal dataset we recommend the `examination_date` column as the time axis.

In [ ]:
indcols = ["donor_et_id_et"]
data = data.sort_index(axis=1).sort_values(indcols + ["examination_date"], axis=0)
data = data.set_index(indcols)

In [ ]:
class DonorPostmortemExamination(SpenderID):
    communicated_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Communicated date",
        description="Date when the lab result was communicated",
    )
    examination_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Examination date",
        description="Date when the lab measurements were conducted",
    )
    examination_result: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Examination result",
        description="What was the result of the examination?",
        isin=["unauffällig", "auffällig", "pathologisch", "grenzwertig"],
    )
    examination_type: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Examination type",
        description="What was examined",
        isin=[
            "CT Thorax",
            "CT Abdomen",
            "Röntgen-Thorax",
            "Ultraschall Abdomen",
            "Bronchoskopie",
            "Ultraschall Herz",
            "EKG",
            "Koronarangiografie",
            "Körperliche Untersuchung",
            "CT Schädel",
            "Sonstige",
            "Coloskopie",
            "Gastroskopie",
            "Lungenszintigrafie",
            "Duodenoskopie",
        ],
    )
    result_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Result date",
        description="Date when the lab result was generated",
    )

    class Config:
        title = "Donor Postmortem Examination Dataset"
        description = "Each row represents a examination of the deceased donor. The data is based on the 'element_spender_postmortem_medikation.csv' file. It contains data from the DSO and ET."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(DonorPostmortemExamination, data)

In [ ]:
DonorPostmortemExamination.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    DonorPostmortemExamination.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)